# AI vs Real Face Detector — Run 2 Full Hybrid Colab Training
## FFHQ + StyleGAN2 + Diffusion

Consolidated training notebook for the `full_hybrid` architecture:

```text
Face
 ├── EfficientNet
 ├── Physics
 ├── PRNU
 └── ViT / Semantic
          ↓
     Gated Fusion
          ↓
       Classifier
          ↓
     REAL / AI / UNCERTAIN
```

**GPU:** T4 recommended.

The notebook first verifies the repository and explicit `train/val/test` dataset, then runs a one-epoch smoke test before the 15-epoch training run.


In [ ]:
# 1. GPU check
import torch

assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)


In [ ]:
# 2. Clone the UPDATED repository
%cd /content
!rm -rf /content/ai-vs-real-face-detector

# This is the repository URL used by the original uploaded training notebook.
!git clone --branch master --single-branch https://github.com/Algorithm-bot/ai-vs-real-face-detector.git /content/ai-vs-real-face-detector

REPO = "/content/ai-vs-real-face-detector"
PROJECT = f"{REPO}/ai-vs-real-face-detector"

!grep -n "full_hybrid" "{PROJECT}/src/train.py" | head -20


In [ ]:
# 3. Mount Drive and define paths
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/ai-vs-real-face-detector")
DATA_DIR = DRIVE_ROOT / "data"
OUTPUT_DIR = DRIVE_ROOT / "models" / "run2_full_hybrid"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", DRIVE_ROOT)
print("DATA:", DATA_DIR)
print("OUTPUT:", OUTPUT_DIR)


In [ ]:
# 4. Install project dependencies
%cd /content/ai-vs-real-face-detector/ai-vs-real-face-detector
!pip -q install -r requirements.txt


In [ ]:
# 5. Verify the explicit train/val/test dataset
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def count_images(path):
    return sum(
        1 for p in Path(path).rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

counts = {}
for split in ["train", "val", "test"]:
    counts[split] = {}
    for label in ["real", "fake"]:
        p = DATA_DIR / split / label
        assert p.exists(), f"Missing dataset directory: {p}"
        counts[split][label] = count_images(p)
        print(f"{split}/{label}: {counts[split][label]}")

expected = {
    "train": {"real": 3500, "fake": 3500},
    "val": {"real": 750, "fake": 750},
    "test": {"real": 750, "fake": 750},
}
assert counts == expected, f"Dataset counts do not match expected Run 2 counts: {counts}"

print("Dataset structure and counts OK.")


## 6. Full-hybrid smoke test

Run this first. It checks the complete multimodal path before spending GPU time on the full run.

If it fails with:
- **CUDA out of memory:** reduce batch size to 1.
- **Missing dependency/model:** fix that before full training.
- **Tensor shape mismatch:** stop and inspect the traceback.
- **PRNU/ViT extraction error:** do not replace features with zeros.


In [ ]:
# 6. Smoke test: 1 epoch, tiny batch
%cd /content/ai-vs-real-face-detector/ai-vs-real-face-detector

!python "{PROJECT}/src/train.py"     --mode full_hybrid     --data-dir "{DATA_DIR}"     --output-dir "{OUTPUT_DIR}/smoke_test"     --epochs 1     --batch-size 2     --num-workers 0     --fusion-mode gated


## 7. Run 2 — Full Hybrid Training

**Start with batch size 16 on a T4.**

The model uses:
- EfficientNet features
- physics features
- PRNU features
- ViT/semantic features

No placeholder vectors.


In [ ]:
# 7. Full training
%cd /content/ai-vs-real-face-detector/ai-vs-real-face-detector

!python "{PROJECT}/src/train.py"     --mode full_hybrid     --data-dir "{DATA_DIR}"     --output-dir "{OUTPUT_DIR}"     --epochs 15     --batch-size 16     --num-workers 2     --fusion-mode gated


In [ ]:
# 8. Check generated artifacts and checkpoint
import json

for p in OUTPUT_DIR.rglob("*"):
    if p.is_file():
        print(p)

checkpoint = OUTPUT_DIR / "full_hybrid_best.pt"
history = OUTPUT_DIR / "full_hybrid_history.json"

print("\nCheckpoint exists:", checkpoint.exists(), checkpoint)
print("History exists:", history.exists(), history)

assert checkpoint.exists(), "full_hybrid_best.pt was not created."


In [ ]:
# 9. Inspect training history and held-out test metrics
if history.exists():
    with open(history) as f:
        h = json.load(f)

    print(json.dumps(h, indent=2)[:8000])

    print("\nAvailable history keys:", list(h.keys()))
    for key in [
        "test_accuracy",
        "test_precision",
        "test_recall",
        "test_f1",
        "test_roc_auc",
    ]:
        if key in h:
            print(f"{key}: {h[key]}")
else:
    print("History file not found:", history)


## 10. Optional attention-fusion comparison

Run this only **after** the gated model completes successfully. It provides the requested gated-vs-attention ablation.

The held-out test split should remain untouched during model selection.


In [ ]:
# Uncomment only after the gated model has completed successfully.

# !python "{PROJECT}/src/train.py" \
#     --mode full_hybrid \
#     --data-dir "{DATA_DIR}" \
#     --output-dir "{OUTPUT_DIR}/attention_fusion" \
#     --epochs 15 \
#     --batch-size 16 \
#     --num-workers 2 \
#     --fusion-mode attention


## 11. Download the trained checkpoint

The checkpoint is already stored on Google Drive. This cell downloads a local copy if needed.


In [ ]:
from google.colab import files

assert checkpoint.exists(), "Train the model first."
files.download(str(checkpoint))


# Run 2 complete

Training data:

**Real:** FFHQ

**Fake:**
- StyleGAN2
- Stable Diffusion v1.5 text-to-image
- Stable Diffusion Inpainting

Next experiments:
1. Compare `full_hybrid_best.pt` against the old `hybrid_best.pt`.
2. Run the gated-vs-attention ablation.
3. Test on an unseen generator for cross-generator generalization.
4. Calibrate on validation data before reporting final test confidence.
